In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
#from tqdm import tqdm
#import re
import os
#from tqdm.notebook import tqdm
from tqdm.auto import tqdm
from collections import defaultdict
from urllib.request import urlretrieve


In [2]:
print('Current working directory is', os.getcwd())

# downloading and combining data from excel sheets
#url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx"
#urlretrieve(url, "online_retail_II.xlsx")

#df_2010 = pd.read_excel("online_retail_II.xlsx", sheet_name="Year 2009-2010")
#df_2011 = pd.read_excel("online_retail_II.xlsx", sheet_name="Year 2010-2011")

#df_total = pd.concat([df_2010, df_2011], ignore_index=True)

#df_total.to_csv("data/online_retail.csv", index=False)

# reading previously saved csv, which contains combined data of two excel sheets
df_total = pd.read_csv("data/online_retail.csv")

Current working directory is /Users/taniazamansarna/projects/customer-clv


In [3]:
df_total.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [4]:
df_total.head(4)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom


**Changing data type of datetime, and renaming Customer ID column**

In [5]:
df_total['InvoiceDate'] = pd.to_datetime(df_total['InvoiceDate'])
df_total = df_total.rename(columns={"Customer ID": "Customer_ID"})

## Splitting the data into calibration and holdout periods

**5 December 2010 is selected as the split date.**

- **Calibration period:** Used to estimate customer behavior and fit the predictive models. Transactions **on or before 5 December 2010** belong to the **calibration period**.
- **Holdout period:** Used only for evaluation. Transactions in this period are treated as unseen future data and are used to assess how well the models generalize.
Transactions **after 5 December 2010** belong to the **holdout period**.

In [6]:
# Split date
split_date = pd.Timestamp("2010-12-05")

# Calibration datasets
df_calib = df_total[df_total["InvoiceDate"] <= split_date].copy()
df_calib.to_csv('data/calibration_raw.csv', index=False)

# Holdout datasets
df = df_total[df_total["InvoiceDate"] > split_date].copy()

print(f"Calibration transactions: {len(df_calib)}")
print(f"Holdout transactions    : {len(df)}")

Calibration transactions: 517776
Holdout transactions    : 549595


# CLEANING HOLDOUT DATA

**Handling Data types and uppercasing string data**

In [7]:
df['Invoice'] = df['Invoice'].astype(str).str.upper().str.strip()
df["Customer_ID"] = df["Customer_ID"].astype("Int64")
df['StockCode'] = df['StockCode'].astype(str).str.upper().str.strip()
df['Description'] = df['Description'].astype(str).str.upper().str.strip()
df['Country'] = df['Country'].astype(str).str.upper().str.strip()
df.info()

<class 'pandas.DataFrame'>
Index: 549595 entries, 510357 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      549595 non-null  str           
 1   StockCode    549595 non-null  str           
 2   Description  548118 non-null  str           
 3   Quantity     549595 non-null  int64         
 4   InvoiceDate  549595 non-null  datetime64[us]
 5   Price        549595 non-null  float64       
 6   Customer_ID  411375 non-null  Int64         
 7   Country      549595 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 38.3 MB


## Handling Returned Orders

Returned orders were identified by matching each return to an earlier purchase from the same customer with the same product, quantity, and price. After matching, all returned transactions were removed, and only completed purchases were kept for the analysis.

The dataset contains both completed purchases and returned items. To identify and remove returned transactions, the data is first sorted chronologically by `InvoiceDate`. Each transaction is then processed in time order.

A purchase is uniquely identified by the combination of:

- `Customer_ID`
- `StockCode`
- `Quantity` (absolute value)
- `Price`

For each positive-quantity transaction (purchase), its position is stored as an unmatched sale. When a negative-quantity transaction (return) is encountered, the algorithm searches for a corresponding unmatched purchase with the same customer, product, quantity, and price.

The matching process assigns one of the following statuses:

- **completed** – A purchase with no matching return.
- **cancelled** – A purchase that was later returned.
- **return: purchase found** – A return successfully matched to a previous purchase.
- **return: purchase not found** – A return with no corresponding purchase in the dataset.

After all transactions have been processed, only transactions labeled **completed** are retained for subsequent analysis. The temporary `status` column is then removed.

In [8]:
# Both negative price and negative quantity mean cancellation of orders; 
# I standardize cancellation of order by considering Quantity -ve and price positive
df['Quantity'] = df['Quantity'] * np.sign(df['Price'])
df['Price'] = np.abs(df['Price'])

# Sort by time
df = df.sort_values("InvoiceDate").reset_index(drop=True)

# Status column
df["status"] = pd.NA

# ---------------------------------------------------
# Matching structure:
# key = (Customer_ID, StockCode, abs(Quantity), Price)
# value = list of unmatched positive row positions
# ---------------------------------------------------
open_sales = defaultdict(list)

# Use tqdm for progress tracking
for i, row in tqdm(
    enumerate(df.itertuples(index=False)),
    total=len(df),
    desc="Matching returns"
):
    key = (
        row.Customer_ID,
        row.StockCode,
        abs(row.Quantity),
        row.Price
    )
    if row.Quantity > 0:
        open_sales[key].append(i)

    elif row.Quantity < 0:
        if open_sales[key]:
            match_i = open_sales[key].pop()
            df.at[match_i, "status"] = "cancelled"
            df.at[i, "status"] = "return: purchase found"
        else:
            df.at[i, "status"] = "return: purchase not found"

# Any remaining positive rows are completed purchases
df.loc[(df["Quantity"] > 0) & (df["status"].isna()), "status"] = "completed"

df['status'] = df['status'].astype(str)

## Keeping only completed orders
df = df[df['status']=='completed']
#df_return = df[df['status'] == 'return: purchase found']

# droping status column before saving to csv
df.drop(columns = 'status', inplace=True)
#df_return.drop(columns = 'status', inplace=True)

Matching returns:   0%|          | 0/549595 [00:00<?, ?it/s]

In [9]:
# Deleting lines that corresponds to a test transaction, or to some fee
df = df[~(df["Description"]=="CHECKED")]
df = df[~(df["Description"]=="CHECK")]
df = df[~(df["Description"]=="CHECK?")]
df = df[~(df["Description"]=="STOCK CHECK")]
df = df[~(df["Description"]=="CARRIAGE")]

df = df[~df["Description"].str.contains("damaged", case=False, na=False)]
df = df[~df["Description"].str.contains("BANK CHARGES", case=False, na=False)]
df = df[~df["Description"].str.contains("DISCOUNT", case=False, na=False)]
df = df[~df["Description"].str.contains("POSTAGE", case=False, na=False)]
df = df[~df["Description"].str.contains("MANUAL", case=False, na=False)]
df = df[~df["Description"].str.contains("ADJUST", case=False, na=False)]
df = df[~df["Description"].str.contains("test", case=False, na=False)]
df = df[~df["Description"].str.contains("lost", case=False, na=False)]
df = df[~df["Description"].str.contains("samples", case=False, na=False)]

# 5. Remove administrative/service stock codes
non_product_codes = ['DOT', 'POST', 'C2', 'M', 'BANK CHARGES', 'PADS', 'CRUK']
df = df[~df['StockCode'].isin(non_product_codes)]

df = df[~df["StockCode"].str.contains("test", case=False, na=False)]
df = df[~df["StockCode"].str.contains("fee", case=False, na=False)]

## BASIC CLEANING

**Handling column mismatch**

In [10]:
### Checking whether rows corresponds to a same invoice referes to a same Customer_ID 

# Count unique Customer_ID  per Invoice
# dropna=False means missing Customer_ID is counted as its own value
invoice_check = (
    df.groupby('Invoice')
      .agg(
          n_customers=('Customer_ID', lambda x: x.nunique(dropna=False))
      )
      .reset_index()
)

# Invoices where same invoice has different customer 
bad_invoices = invoice_check[(invoice_check['n_customers'] > 1)]

# Save all original rows belonging to those problematic invoices
invoice_exceptions = (
    df[df['Invoice'].isin(bad_invoices['Invoice'])]
    .sort_values(['Invoice', 'Customer_ID'])
)


if bad_invoices.shape[0] == 0:
    print('No mismatch found between invoice and Customer_ID.  ✅')
else:
    print(f'{bad_invoices.shape[0]} mismatches observed between invoice and Customer_ID. Some mismatches as examples are:')
    print(invoice_exceptions[['Invoice', 'Customer_ID', 'Country']].drop_duplicates().head(5))

No mismatch found between invoice and Customer_ID.  ✅


**Handling duplicates**

In [11]:
# Number of duplicate rows (considering all columns)
n_duplicates = df.duplicated().sum()

if n_duplicates == 0:
    print("✅ No duplicate rows found.")
else:
    print(f"❌ Found {n_duplicates:,} duplicate rows.")

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)

    print(f"Deleted {n_duplicates:,} duplicate rows.")
    print(f"Remaining rows: {len(df):,}")

❌ Found 19,693 duplicate rows.
Deleted 19,693 duplicate rows.
Remaining rows: 512,647


In [12]:
# Standardarizing the product description
collapsed = (
    df.groupby(['StockCode', 'Description'])
      .size()
      .rename('count')
      .reset_index()
)

collapsed['rank'] = (
    collapsed.groupby('StockCode')['count']
             .rank(method='first', ascending=False)
)

collapsed = collapsed[collapsed['rank'] == 1]

df = df.merge(collapsed, on='StockCode', how='left', suffixes=('', '_top'))
df['Description'] = df['Description_top']
df = df.drop(columns=['Description_top', 'rank', 'count'])

## Saving cleaned holdout data

**Saving Transaction Data**

In [13]:
df.to_csv('data/holdout_transaction.csv', index=False)

**Saving Product catalog**

In [14]:
tmp = df.groupby("StockCode").agg(minprice=("Price","min"), maxprice=("Price","max"), avgprice=("Price","mean"), std_of_price=("Price","std"))
desc = df.groupby(["StockCode","Description"]).size().reset_index(name="n").sort_values(["StockCode","n"], ascending=[True,False]).drop_duplicates("StockCode").set_index("StockCode")[["Description"]]
df_product = desc.join(tmp).reset_index()
df_product["std_of_price"] = df_product["std_of_price"].fillna(0)
df_product.head()

#df.drop(columns='Description', inplace=True)

# Saving product catalog
df_product.to_csv('data/prod_catalog_holdout.csv', index=False)